In [59]:
import os
import numpy as np
import pandas as pd
import xarray as xr
from scipy.signal import detrend
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

INPUT          = "/home/26n5005/climate-data/mswep_pr_daily_brunei_1980_2025.nc"
LANDMASK       = "/home/26n5005/lecture2/brunei_landmask_mswep_0p1deg.nc"
NMODES         = 10
MASK_THRESHOLD = 0.5


In [60]:
# HELPER FUNCTIONS

def find_coord_name(obj, candidates):
    """Return the first matching coordinate/dimension name."""
    for name in candidates:
        if name in obj.coords or name in obj.dims:
            return name
    return None


def choose_precip_variable(ds):
    """Choose the most likely precipitation variable."""
    preferred = [
        "pr",
        "precipitation",
        "precip",
        "rain",
        "rainfall"
    ]

    for name in preferred:
        if name in ds.data_vars:
            return name

    # Prefer a variable containing time + two spatial dimensions.
    for name, da in ds.data_vars.items():
        if "time" in da.dims and da.ndim >= 3:
            return name

    return list(ds.data_vars)[0]


def choose_mask_variable(ds):
    """Choose the most likely land-mask variable."""
    preferred = [
        "landmask",
        "land_mask",
        "mask",
        "lmask",
        "brunei_landmask",
        "mask_binary"
    ]

    for name in preferred:
        if name in ds.data_vars:
            return name

    # Prefer a two-dimensional variable without time.
    for name, da in ds.data_vars.items():
        if "time" not in da.dims and da.ndim == 2:
            return name

    return list(ds.data_vars)[0]

In [61]:
# STEP 1: READ DAILY PRECIPITATION

ds = xr.open_dataset(INPUT)

varname = choose_precip_variable(ds)
pr_daily = ds[varname]

lat_name = find_coord_name(pr_daily, ["lat", "latitude"])
lon_name = find_coord_name(pr_daily, ["lon", "longitude"])

print(f"Precipitation variable : {varname}")
print(f"Dimensions             : {pr_daily.dims}")
print(f"Shape                  : {pr_daily.shape}")
print(f"Latitude coordinate    : {lat_name}")
print(f"Longitude coordinate   : {lon_name}")
print(f"Input units            : {pr_daily.attrs.get('units', 'not specified')}")

Precipitation variable : precipitation
Dimensions             : ('time', 'lat', 'lon')
Shape                  : (16802, 13, 15)
Latitude coordinate    : lat
Longitude coordinate   : lon
Input units            : mm d-1


In [62]:
# STEP 1B: RESTRICT MSWEP TO THE STUDY PERIOD

START_DATE = "1981-09-01"
END_DATE = "2025-12-31"

pr_daily = pr_daily.sel(
    time=slice(START_DATE, END_DATE)
)

print(
    f"MSWEP period: {str(pr_daily.time.values[0])[:10]} "
    f"to {str(pr_daily.time.values[-1])[:10]}"
)
print(f"Daily time steps: {pr_daily.sizes['time']}")

MSWEP period: 1981-09-01 to 2025-12-31
Daily time steps: 16193


In [63]:
# STEP 2: READ AND ALIGN BRUNEI LAND MASK

mask_ds = xr.open_dataset(LANDMASK)
mask_varname = choose_mask_variable(mask_ds)
mask_raw = mask_ds[mask_varname].squeeze(drop=True)

mask_lat_name = find_coord_name(mask_raw, ["lat", "latitude"])
mask_lon_name = find_coord_name(mask_raw, ["lon", "longitude"])

print(f"Mask variable          : {mask_varname}")
print(f"Mask dimensions        : {mask_raw.dims}")
print(f"Mask shape             : {mask_raw.shape}")
print(f"Mask latitude          : {mask_lat_name}")
print(f"Mask longitude         : {mask_lon_name}")

# Rename mask coordinates/dimensions to match the precipitation file.
rename_map = {}

if mask_lat_name != lat_name:
    rename_map[mask_lat_name] = lat_name
if mask_lon_name != lon_name:
    rename_map[mask_lon_name] = lon_name
if rename_map:
    mask_raw = mask_raw.rename(rename_map)

# Ensure the mask contains only the spatial dimensions.
extra_dims = [
    d for d in mask_raw.dims
    if d not in [lat_name, lon_name]
]

for d in extra_dims:
    if mask_raw.sizes[d] == 1:
        mask_raw = mask_raw.isel({d: 0}, drop=True)

# Align the mask to the precipitation grid.
# First try exact coordinate selection. If exact values differ only very slightly,
# use nearest-neighbour interpolation onto the precipitation coordinates.
try:
    mask_aligned = mask_raw.sel(
        {
            lat_name: pr_daily[lat_name],
            lon_name: pr_daily[lon_name]
        }
    )

except Exception:
    print(
        "Exact coordinate alignment failed; "
        "using nearest-neighbour alignment to precipitation grid."
    )

    mask_aligned = mask_raw.interp(
        {
            lat_name: pr_daily[lat_name],
            lon_name: pr_daily[lon_name]
        },
        method="nearest"
    )

# Convert the aligned mask to a boolean retained-cell mask.
brunei_mask = (
    np.isfinite(mask_aligned)
    & (mask_aligned > MASK_THRESHOLD)
)

brunei_mask = brunei_mask.astype(bool)
brunei_mask.name = "brunei_landmask"

n_domain_cells = (
    pr_daily.sizes[lat_name]
    * pr_daily.sizes[lon_name]
)

n_mask_cells = int(brunei_mask.sum().item())

print(f"Full precipitation-domain cells : {n_domain_cells}")
print(f"Retained Brunei mask cells       : {n_mask_cells}")

# Save the actual mask used by this analysis.
mask_used_file = "brunei_landmask_used_for_EOF_on_mswep_grid.nc"

brunei_mask.astype(np.int8).to_dataset(
    name="brunei_landmask"
).to_netcdf(mask_used_file)

print(f"Saved aligned mask: {mask_used_file}")

# STEP 3: APPLY LAND MASK TO DAILY PRECIPITATION
pr_daily_masked = pr_daily.where(brunei_mask)

Mask variable          : brunei_landmask
Mask dimensions        : ('lat', 'lon')
Mask shape             : (13, 15)
Mask latitude          : lat
Mask longitude         : lon
Full precipitation-domain cells : 195
Retained Brunei mask cells       : 52
Saved aligned mask: brunei_landmask_used_for_EOF_on_mswep_grid.nc


In [64]:
# STEP 4: CALCULATE MONTHLY PRECIPITATION TOTALS

monthly_resampler = pr_daily_masked.resample(time="MS")

# Sum valid daily precipitation.
# pr_monthly = monthly_resampler.sum(
#     dim="time",
#     skipna=True
# )

# Mean valid daily precipitation.
pr_monthly = monthly_resampler.mean(
    dim="time",
    skipna=True
)

monthly_valid_count = monthly_resampler.count(dim="time")
pr_monthly = pr_monthly.where(monthly_valid_count > 0)

# Reapply Brunei mask explicitly.
pr_monthly = pr_monthly.where(brunei_mask)

pr_monthly.name = "pr_monthly"
pr_monthly.attrs["long_name"] = (
    "Monthly accumulated precipitation over Brunei land-mask grid cells"
)
pr_monthly.attrs["units"] = "mm month-1"

monthly_file = "mswep_pr_monthly_brunei_landmask_1980_2025.nc"
pr_monthly.to_netcdf(monthly_file)

print(f"Number of months: {pr_monthly.sizes['time']}")
print(f"Saved: {monthly_file}")

Number of months: 532
Saved: mswep_pr_monthly_brunei_landmask_1980_2025.nc


In [65]:
# STEP 5: CALCULATE MONTHLY CLIMATOLOGY

monthly_climatology = pr_monthly.groupby(
    "time.month"
).mean(
    dim="time",
    skipna=True
)

monthly_climatology = monthly_climatology.where(brunei_mask)

monthly_climatology.name = "pr_monthly_climatology"
monthly_climatology.attrs["long_name"] = (
    "Climatological monthly precipitation over Brunei land-mask grid cells"
)

monthly_climatology.attrs["units"] = "mm month-1"
clim_file = "mswep_pr_monthly_climatology_brunei_landmask_1980_2025.nc"
monthly_climatology.to_netcdf(clim_file)
print(f"Saved: {clim_file}")

Saved: mswep_pr_monthly_climatology_brunei_landmask_1980_2025.nc


In [66]:
# STEP 6: CALCULATE MONTHLY ANOMALIES

monthly_anomaly = (pr_monthly.groupby("time.month") - monthly_climatology)
monthly_anomaly = monthly_anomaly.where(brunei_mask)

monthly_anomaly.name = "pr_anomaly"
monthly_anomaly.attrs["long_name"] = (
    "Monthly precipitation anomaly over Brunei land-mask grid cells"
)
monthly_anomaly.attrs["units"] = "mm month-1"

anom_file = "mswep_pr_monthly_anomaly_brunei_landmask_1980_2025.nc"
monthly_anomaly.to_netcdf(anom_file)
print(f"Saved: {anom_file}")

Saved: mswep_pr_monthly_anomaly_brunei_landmask_1980_2025.nc


In [67]:
# STEP 7: DETREND MONTHLY ANOMALIES

spatial_dims = [
    d for d in monthly_anomaly.dims
    if d != "time"
]

anom = monthly_anomaly.transpose("time", *spatial_dims)
ntime = anom.sizes["time"]
original_shape = anom.shape
X_full = anom.values.reshape(ntime, -1)

# Flatten the Brunei land mask in exactly the same spatial order as the anomaly.
mask_2d = brunei_mask.transpose(*spatial_dims).values.astype(bool)
mask_flat = mask_2d.reshape(-1)

# A cell is eligible only if:
#   1. it belongs to the Brunei mask, AND
#   2. it has a complete monthly anomaly record.
complete_record = np.all(np.isfinite(X_full), axis=0)
valid_space = (mask_flat & complete_record)
X = X_full[:, valid_space]

print(f"Full-domain spatial cells          : {X_full.shape[1]}")
print(f"Brunei land-mask cells             : {mask_flat.sum()}")
print(f"Complete Brunei cells used for EOF : {X.shape[1]}")
print(f"Monthly time steps                 : {ntime}")

# Detrend independently at every retained Brunei grid cell.
X_detrended = detrend(
    X,
    axis=0,
    type="linear"
)

# Restore the detrended anomalies to the FULL precipitation grid.
# Outside-mask / unusable cells remain NaN.
X_dt_full = np.full(
    X_full.shape,
    np.nan,
    dtype=np.float64
)

X_dt_full[:, valid_space] = X_detrended
anom_detrended = xr.DataArray(
    X_dt_full.reshape(original_shape),
    coords=anom.coords,
    dims=anom.dims,
    name="pr_anomaly_detrended"
)

anom_detrended.attrs["long_name"] = (
    "Detrended monthly precipitation anomaly over Brunei land-mask grid cells"
)
anom_detrended.attrs["units"] = "mm month-1"

dt_file = "mswep_pr_monthly_anomaly_detrended_brunei_landmask_1980_2025.nc"
anom_detrended.to_netcdf(dt_file)
print(f"Saved: {dt_file}")

Full-domain spatial cells          : 195
Brunei land-mask cells             : 52
Complete Brunei cells used for EOF : 52
Monthly time steps                 : 532
Saved: mswep_pr_monthly_anomaly_detrended_brunei_landmask_1980_2025.nc


In [68]:
# STEP 8: APPLY LATITUDE WEIGHTING
lat = anom_detrended[lat_name]
weights = np.sqrt(np.cos(np.deg2rad(lat)))
weighted = anom_detrended * weights

In [69]:
# STEP 9: PREPARE SVD MATRIX AND SPLIT TRAIN / TEST USING ONLY BRUNEI MASK CELLS

weighted = weighted.transpose("time", *spatial_dims)
X_weighted_full = weighted.values.reshape(ntime, -1)

# Use exactly the valid Brunei cells identified before detrending.
X_svd = X_weighted_full[:, valid_space]


# Chronological split: last 60 months are test.
TEST_MONTHS = 60
split = ntime - TEST_MONTHS

X_train = X_svd[:split].copy()
X_test = X_svd[split:].copy()


# Remove temporal mean using TRAIN data only.
train_mean = X_train.mean(axis=0, keepdims=True)

X_train = X_train - train_mean
X_test = X_test - train_mean

print(f"Train matrix: {X_train.shape}")
print(f"Test matrix:  {X_test.shape}")

Train matrix: (472, 52)
Test matrix:  (60, 52)


In [70]:
# STEP 10: PERFORM SVD

U, singular_values, VT = np.linalg.svd(X_train, full_matrices=False)
nmodes_available = min(NMODES, VT.shape[0])

print(f"Available SVD modes : {VT.shape[0]}")
print(f"Saving modes        : {nmodes_available}")

Available SVD modes : 52
Saving modes        : 10


In [71]:
# STEP 11: EIGENVALUES AND EXPLAINED VARIANCE

eigenvalues = (singular_values ** 2) / (ntime - 1)

explained_variance = (
    eigenvalues
    / np.sum(eigenvalues)
    * 100.0
)

cumulative_variance = np.cumsum(explained_variance)
for i in range(nmodes_available):
    print(
        f"EOF{i+1:02d}: "
        f"{explained_variance[i]:7.3f}%  "
        f"Cumulative = {cumulative_variance[i]:7.3f}%"
    )

EOF01:  86.670%  Cumulative =  86.670%
EOF02:  10.951%  Cumulative =  97.621%
EOF03:   0.895%  Cumulative =  98.516%
EOF04:   0.436%  Cumulative =  98.952%
EOF05:   0.273%  Cumulative =  99.224%
EOF06:   0.149%  Cumulative =  99.373%
EOF07:   0.133%  Cumulative =  99.506%
EOF08:   0.081%  Cumulative =  99.587%
EOF09:   0.077%  Cumulative =  99.664%
EOF10:   0.048%  Cumulative =  99.713%


In [72]:
# STEP 12: CALCULATE PRINCIPAL COMPONENTS (train)

PC_train = (
    U[:, :nmodes_available]
    * singular_values[:nmodes_available]
)

PC_test = X_test @ VT[:nmodes_available, :].T

print("PC_train shape:", PC_train.shape)
print("PC_test shape:", PC_test.shape)

PC_train shape: (472, 10)
PC_test shape: (60, 10)


In [73]:
# STEP 13: RECONSTRUCT EOFs ON FULL PRECIPITATION GRID

nspace_full = X_full.shape[1]

# Start with full-domain NaNs.
EOF_weighted_flat = np.full(
    (nmodes_available, nspace_full),
    np.nan,
    dtype=np.float64
)

# Put SVD EOF loadings back ONLY at Brunei grid cells used in the analysis.
EOF_weighted_flat[:, valid_space] = VT[
    :nmodes_available,
    :
]

EOF_shape = (
    nmodes_available,
    *[
        anom_detrended.sizes[d]
        for d in spatial_dims
    ]
)

EOF_weighted_full = EOF_weighted_flat.reshape(
    EOF_shape
)

# Convert weighted EOFs back to physical spatial loadings.
lat_axis = spatial_dims.index(lat_name)

weight_shape = [1] * len(spatial_dims)
weight_shape[lat_axis] = len(lat)

w_array = weights.values.reshape(
    weight_shape
)

EOF_physical_full = (
    EOF_weighted_full
    / w_array
)

# Explicitly restore the Brunei land mask on every EOF map.
mask_broadcast = mask_2d[None, :, :]

EOF_physical_full = np.where(
    mask_broadcast,
    EOF_physical_full,
    np.nan
)


In [74]:
# STEP 15A: SAVE PC TIME SERIES — TRAIN

pc_train_da = xr.DataArray(
    PC_train,
    dims=["time", "mode"],
    coords={
        "time": anom_detrended.time[:len(PC_train)],
        "mode": np.arange(1, PC_train.shape[1] + 1)
    },
    name="PC"
)

pc_train_da.attrs["long_name"] = (
    "Training principal component time series from Brunei land-mask EOF analysis"
)
pc_train_da.attrs["mask_file"] = str(LANDMASK)
pc_train_da.attrs["number_of_EOF_cells"] = int(valid_space.sum())

TRAIN_PC_FILE = "mswep_brunei_landmask_PC_train.nc"
pc_train_da.to_netcdf(TRAIN_PC_FILE)
print(f"Saved: {TRAIN_PC_FILE}")


# STEP 15B: SAVE PC TIME SERIES — TEST

pc_test_da = xr.DataArray(
    PC_test,
    dims=["time", "mode"],
    coords={
        "time": anom_detrended.time[len(PC_train):],
        "mode": np.arange(1, PC_test.shape[1] + 1)
    },
    name="PC"
)

pc_test_da.attrs["long_name"] = (
    "Test principal component time series projected onto training EOFs"
)
pc_test_da.attrs["mask_file"] = str(LANDMASK)
pc_test_da.attrs["number_of_EOF_cells"] = int(valid_space.sum())

TEST_PC_FILE = "mswep_brunei_landmask_PC_test.nc"
pc_test_da.to_netcdf(TEST_PC_FILE)

print(f"Saved: {TEST_PC_FILE}")

Saved: mswep_brunei_landmask_PC_train.nc
Saved: mswep_brunei_landmask_PC_test.nc


In [56]:
# STEP 16: SAVE EOF STATISTICS

stats = xr.Dataset(
    data_vars={
        "eigenvalue": (
            "mode",
            eigenvalues[:nmodes_available]
        ),
        "explained_variance": (
            "mode",
            explained_variance[:nmodes_available]
        ),
        "cumulative_variance": (
            "mode",
            cumulative_variance[:nmodes_available]
        )
    },
    coords={
        "mode": np.arange(
            1,
            nmodes_available + 1
        )
    }
)

stats["explained_variance"].attrs["units"] = "%"
stats["cumulative_variance"].attrs["units"] = "%"

stats.attrs["mask_file"] = LANDMASK
stats.attrs["mask_threshold"] = MASK_THRESHOLD
stats.attrs["number_of_mask_cells"] = n_mask_cells
stats.attrs["number_of_EOF_cells"] = int(valid_space.sum())

stats_file = "mswep_brunei_landmask_EOF_statistics_10.nc"

stats.to_netcdf(
    stats_file
)

csv_file = "mswep_brunei_landmask_EOF_statistics_10.csv"

pd.DataFrame(
    {
        "mode":
            np.arange(
                1,
                nmodes_available + 1
            ),
        "eigenvalue":
            eigenvalues[
                :nmodes_available
            ],
        "explained_variance_percent":
            explained_variance[
                :nmodes_available
            ],
        "cumulative_variance_percent":
            cumulative_variance[
                :nmodes_available
            ]
    }
).to_csv(
    csv_file,
    index=False
)

print(f"Saved: {stats_file}")
print(f"Saved: {csv_file}")

Saved: mswep_brunei_landmask_EOF_statistics_10.nc
Saved: mswep_brunei_landmask_EOF_statistics_10.csv


In [57]:
# STEP 17: CREATE SCREE PLOT

modes = np.arange(
    1,
    nmodes_available + 1
)

fig, ax1 = plt.subplots(
    figsize=(9, 6)
)

line1, = ax1.plot(
    modes,
    explained_variance[:nmodes_available],
    marker="o",
    linewidth=1.8,
    markersize=6,
    label="Individual variance"
)

ax1.set_xlabel(
    "EOF mode",
    fontsize=12
)

ax1.set_ylabel(
    "Explained variance (%)",
    fontsize=12
)

ax1.set_xticks(
    modes
)

ax1.grid(
    True,
    linestyle="--",
    alpha=0.35
)

for mode, var in zip(
    modes,
    explained_variance[:nmodes_available]
):
    ax1.annotate(
        f"{var:.1f}%",
        xy=(mode, var),
        xytext=(0, 7),
        textcoords="offset points",
        ha="center",
        fontsize=9
    )

ax2 = ax1.twinx()

line2, = ax2.plot(
    modes,
    cumulative_variance[:nmodes_available],
    marker="s",
    linewidth=1.5,
    linestyle="--",
    label="Cumulative variance"
)

ax2.set_ylabel(
    "Cumulative explained variance (%)",
    fontsize=12
)

ax2.set_ylim(
    0,
    105
)

for threshold in [70, 80, 90]:
    ax2.axhline(
        threshold,
        linewidth=0.8,
        linestyle=":",
        alpha=0.6
    )

ax1.legend(
    [line1, line2],
    ["Individual variance", "Cumulative variance"],
    loc="center right",
    frameon=True
)

plt.title(
    "Scree Plot: Monthly MSWEP Precipitation EOFs over Brunei Land Grid",
    fontsize=13,
    fontweight="bold"
)

plt.tight_layout()
SCREE_FILE = "mswep_brunei_landmask_EOF_scree_plot.png"
plt.savefig(
    SCREE_FILE,
    dpi=300,
    bbox_inches="tight"
)
plt.close()
print(f"Saved: {SCREE_FILE}")

Saved: mswep_brunei_landmask_EOF_scree_plot.png


In [58]:
# STEP 18: PLOT EOF1-EOF3 AND PC1-PC3 — TRAIN / TEST

fig = plt.figure(figsize=(16, 12))
lon_values = anom_detrended[lon_name].values
lat_values = anom_detrended[lat_name].values

map_extent = [
    float(np.nanmin(lon_values)),
    float(np.nanmax(lon_values)),
    float(np.nanmin(lat_values)),
    float(np.nanmax(lat_values))
]

nplot = min(3, nmodes_available)

# Time coordinates for the chronological split
time_values = anom_detrended.time.values
train_time = time_values[:len(PC_train)]
test_time = time_values[len(PC_train):]

for mode in range(nplot):

    #---------------------------------------------------------------------------
    # LEFT: EOF spatial pattern learned from TRAIN data
    #---------------------------------------------------------------------------

    ax = fig.add_subplot(
        3, 2, 2 * mode + 1,
        projection=ccrs.PlateCarree()
    )

    field = eof_da.isel(mode=mode)

    vmax = np.nanmax(np.abs(field.values))

    if not np.isfinite(vmax) or vmax == 0:
        vmax = 1.0

    im = ax.pcolormesh(
        anom_detrended[lon_name],
        anom_detrended[lat_name],
        field,
        transform=ccrs.PlateCarree(),
        cmap="RdBu_r",
        vmin=-vmax,
        vmax=vmax,
        shading="auto"
    )

    ax.contour(
        anom_detrended[lon_name],
        anom_detrended[lat_name],
        brunei_mask.astype(float),
        levels=[0.5],
        colors="black",
        linewidths=1.0,
        transform=ccrs.PlateCarree()
    )

    ax.set_extent(map_extent, crs=ccrs.PlateCarree())
    ax.coastlines(resolution="10m", linewidth=1.0)
    ax.add_feature(cfeature.BORDERS, linewidth=0.7)

    gl = ax.gridlines(
        draw_labels=True,
        linewidth=0.4,
        alpha=0.5,
        linestyle="--"
    )
    gl.top_labels = False
    gl.right_labels = False

    ax.set_title(
        f"EOF{mode + 1} (train: {explained_variance[mode]:.2f}%)",
        fontsize=13,
        fontweight="bold"
    )

    cb = plt.colorbar(im, ax=ax, shrink=0.85, pad=0.03)
    cb.set_label("EOF loading")

    #---------------------------------------------------------------------------
    # RIGHT: PC time series — TRAIN and TEST
    #---------------------------------------------------------------------------

    ax_pc = fig.add_subplot(3, 2, 2 * mode + 2)

    ax_pc.plot(
        train_time,
        PC_train[:, mode],
        linewidth=0.8,
        label="Train"
    )

    ax_pc.plot(
        test_time,
        PC_test[:, mode],
        linewidth=0.8,
        label="Test"
    )

    ax_pc.axhline(0, color="black", linewidth=0.7)

    # Mark the boundary between train and test
    if len(test_time) > 0:
        ax_pc.axvline(
            test_time[0],
            color="black",
            linestyle="--",
            linewidth=1.0,
            label="Train/test split"
        )

    ax_pc.set_title(f"PC{mode + 1}", fontsize=13, fontweight="bold")
    ax_pc.set_ylabel("PC amplitude")
    ax_pc.grid(True, linestyle="--", alpha=0.3)
    ax_pc.legend()

fig.suptitle(
    "EOF Analysis of Monthly MSWEP Precipitation over Brunei\n"
    "EOFs fitted on training data; test PCs projected onto training EOFs",
    fontsize=15,
    fontweight="bold",
    y=0.995
)

plt.tight_layout(rect=[0, 0, 1, 0.96])

PLOT_FILE = "mswep_brunei_landmask_EOF1-3_PC1-3_train_test.png"
plt.savefig(PLOT_FILE, dpi=300, bbox_inches="tight")
plt.close()
print(f"Saved: {PLOT_FILE}")

Saved: mswep_brunei_landmask_EOF1-3_PC1-3_train_test.png
